In [1]:
# =========================
# XGBoost GPU - One Cell Full Pipeline
# (Đọc CSV từ folder data/, split, train, evaluate, save model)
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from xgboost import XGBClassifier
from joblib import dump

# ---------- 1) Path ----------
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
CSV_PATH = DATA_DIR / "E:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv"  # đổi tên nếu file bạn khác
TARGET = "Diabetes_012"  # đổi nếu cột target bạn khác

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())

# ---------- 2) Load ----------
df = pd.read_csv(CSV_PATH)
assert TARGET in df.columns, f"Không thấy cột target '{TARGET}'. Cột hiện có: {list(df.columns)[:30]}..."

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

print("Data shape:", df.shape)
print("Class distribution:\n", y.value_counts().sort_index())

# ---------- 3) Split ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

# ---------- 4) Preprocess (numeric impute) ----------
num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols)
    ],
    remainder="drop"
)

# ---------- 5) XGBoost GPU ----------
num_classes = int(np.unique(y_train).size)

xgb_clf = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="gpu_hist",      # ✅ GPU train
    predictor="gpu_predictor",   # ✅ GPU predict
    random_state=42
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", xgb_clf)
])

# ---------- 6) Train ----------
model.fit(X_train, y_train)

# ---------- 7) Validate ----------
pred_val = model.predict(X_val)
proba_val = model.predict_proba(X_val)

print("\n===== VALIDATION =====")
print("Val accuracy:", accuracy_score(y_val, pred_val))
print(classification_report(y_val, pred_val, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_val, pred_val))

try:
    auc_val = roc_auc_score(y_val, proba_val, multi_class="ovr")
    print("Val ROC-AUC (OVR):", auc_val)
except Exception as e:
    print("ROC-AUC không tính được (có thể do thiếu class trong split):", e)

# ---------- 8) Test ----------
pred_test = model.predict(X_test)
proba_test = model.predict_proba(X_test)

print("\n===== TEST =====")
print("Test accuracy:", accuracy_score(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test))

try:
    auc_test = roc_auc_score(y_test, proba_test, multi_class="ovr")
    print("Test ROC-AUC (OVR):", auc_test)
except Exception as e:
    print("ROC-AUC không tính được:", e)

# ---------- 9) Save ----------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_diabetes_012_gpu.joblib"
dump(model, out_path)
print("\nSaved model:", out_path)

ROOT: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code
CSV_PATH: E:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv | exists: True
Data shape: (253680, 22)
Class distribution:
 Diabetes_012
0    213703
1      4631
2     35346
Name: count, dtype: int64
Train/Val/Test: (177576, 21) (38052, 21) (38052, 21)


c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [19:07:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [19:07:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:2676: UserWarning: [19:07:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA inst


===== VALIDATION =====
Val accuracy: 0.8504677809313571
              precision    recall  f1-score   support

           0     0.8647    0.9779    0.9178     32056
           1     0.0000    0.0000    0.0000       694
           2     0.5643    0.1912    0.2857      5302

    accuracy                         0.8505     38052
   macro avg     0.4763    0.3897    0.4012     38052
weighted avg     0.8070    0.8505    0.8130     38052

Confusion matrix:
 [[31348     0   708]
 [  619     0    75]
 [ 4288     0  1014]]
Val ROC-AUC (OVR): 0.781414404192866


c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave


===== TEST =====
Test accuracy: 0.8490486702407232
              precision    recall  f1-score   support

           0     0.8646    0.9760    0.9169     32055
           1     0.0000    0.0000    0.0000       695
           2     0.5480    0.1928    0.2852      5302

    accuracy                         0.8490     38052
   macro avg     0.4709    0.3896    0.4007     38052
weighted avg     0.8047    0.8490    0.8121     38052

Confusion matrix:
 [[31286     0   769]
 [  621     0    74]
 [ 4280     0  1022]]
Test ROC-AUC (OVR): 0.7813931831762001

Saved model: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code\models\xgb_diabetes_012_gpu.joblib


c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\letra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [2]:
# =========================
# XGBoost (XGBoost 2.x GPU) + Class Imbalance (sample_weight) - One Cell
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from joblib import dump

from xgboost import XGBClassifier

# ---------- 1) PATH ----------
ROOT = Path.cwd()
DATA_DIR = ROOT.parent / "Data" if (ROOT / "data").exists() is False else (ROOT / "data")  # linh hoạt
# Bạn đã có: Code/ và Data/ cùng cấp => ROOT là Code => ROOT.parent/Data
CSV_PATH = (ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv")
TARGET = "Diabetes_012"

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())

# ---------- 2) LOAD ----------
df = pd.read_csv(CSV_PATH)
assert TARGET in df.columns, f"Không thấy cột target '{TARGET}'."

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

print("Data shape:", df.shape)
print("Class distribution:\n", y.value_counts().sort_index())

# ---------- 3) SPLIT ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

# ---------- 4) PREPROCESS ----------
num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols)
    ],
    remainder="drop"
)

# ---------- 5) CLASS WEIGHTS -> sample_weight ----------
# Trọng số tỉ lệ nghịch với tần suất lớp (giúp model chịu học lớp 1 hơn)
class_counts = y_train.value_counts().to_dict()
n = len(y_train)
num_classes = int(np.unique(y_train).size)

class_weight = {c: n / (num_classes * class_counts[c]) for c in class_counts}
w_train = y_train.map(class_weight).astype(float).values

print("Class_weight:", class_weight)

# ---------- 6) XGBOOST GPU (XGBoost 2.x) ----------
xgb_clf = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",   # ✅ thay gpu_hist
    device="cuda",        # ✅ GPU training
    random_state=42
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", xgb_clf)
])

# Fit với sample_weight (đẩy trọng số vào bước clf)
model.fit(X_train, y_train, clf__sample_weight=w_train)

# ---------- 7) VALIDATION ----------
pred_val = model.predict(X_val)
proba_val = model.predict_proba(X_val)

print("\n===== VALIDATION =====")
print("Val accuracy:", accuracy_score(y_val, pred_val))
print(classification_report(y_val, pred_val, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_val, pred_val))

try:
    print("Val ROC-AUC (OVR):", roc_auc_score(y_val, proba_val, multi_class="ovr"))
except Exception as e:
    print("ROC-AUC không tính được:", e)

# ---------- 8) TEST ----------
pred_test = model.predict(X_test)
proba_test = model.predict_proba(X_test)

print("\n===== TEST =====")
print("Test accuracy:", accuracy_score(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test))

try:
    print("Test ROC-AUC (OVR):", roc_auc_score(y_test, proba_test, multi_class="ovr"))
except Exception as e:
    print("ROC-AUC không tính được:", e)

# ---------- 9) SAVE ----------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_diabetes_012_cuda_weighted.joblib"
dump(model, out_path)
print("\nSaved model:", out_path)

ROOT: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code
CSV_PATH: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv | exists: True
Data shape: (253680, 22)
Class distribution:
 Diabetes_012
0    213703
1      4631
2     35346
Name: count, dtype: int64
Train/Val/Test: (177576, 21) (38052, 21) (38052, 21)
Class_weight: {0: 0.3956896090700037, 2: 2.3923692506668823, 1: 18.257865515114126}

===== VALIDATION =====
Val accuracy: 0.6700042047724167
              precision    recall  f1-score   support

           0     0.9472    0.6816    0.7927     32056
           1     0.0244    0.1527    0.0421       694
           2     0.3327    0.6679    0.4442      5302

    accuracy                         0.6700     38052
   macro avg     0.4348    0.5007    0.4263     38052
weighted avg     0.8447    0.6700    0.7304     38052

Confusion matrix:
 [[21848  3474  6734]
 [  221   106   367]
 [ 

In [4]:
# =========================
# 2-Stage XGBoost GPU (XGB 2.x): FIXED predict_012 indexing
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from xgboost import XGBClassifier
from joblib import dump

# ---- paths ----
ROOT = Path.cwd()
CSV_PATH = ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv"
TARGET = "Diabetes_012"

df = pd.read_csv(CSV_PATH)
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols)],
    remainder="drop"
)

# =========================
# Stage A: Binary (0 vs risk={1,2})
# =========================
yA_train = (y_train != 0).astype(int)  # 1 = risk

neg = int((yA_train == 0).sum())
pos = int((yA_train == 1).sum())
scale_pos_weight = float(neg) / float(pos)

stageA = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=1600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        random_state=42
    ))
])
stageA.fit(X_train, yA_train)

# =========================
# Stage B: (1 vs 2) only on risk samples
# =========================
maskB_train = (y_train != 0)
XB_train = X_train.loc[maskB_train]
yB_train = y_train.loc[maskB_train].map({1:0, 2:1}).astype(int)  # 0=pre, 1=diabetes

countsB = yB_train.value_counts().to_dict()
nB = int(len(yB_train))
class_weightB = {c: nB/(2*countsB[c]) for c in countsB}
wB_train = yB_train.map(class_weightB).values

stageB = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cuda",
        random_state=42
    ))
])
stageB.fit(XB_train, yB_train, clf__sample_weight=wB_train)

# =========================
# Predict 3-class via 2-stage (FIXED)
# =========================
def predict_012(X_in: pd.DataFrame, threshold_risk: float = 0.40) -> np.ndarray:
    p_risk = stageA.predict_proba(X_in)[:, 1]
    is_risk = (p_risk >= threshold_risk)          # numpy bool array

    out = np.zeros(len(X_in), dtype=int)          # default class 0

    if np.any(is_risk):
        X_r = X_in.iloc[np.where(is_risk)[0]]     # chọn rows risk an toàn
        p_b = stageB.predict_proba(X_r)[:, 1]     # 1 => class 2
        out_r = np.where(p_b >= 0.5, 2, 1)        # split 1 vs 2
        out[is_risk] = out_r                      # ✅ FIX: không dùng .values
    return out

# ---- Evaluate ----
for name, X_eval, y_eval in [("VAL", X_val, y_val), ("TEST", X_test, y_test)]:
    pred = predict_012(X_eval, threshold_risk=0.40)
    print(f"\n===== {name} (2-stage) =====")
    print("Accuracy:", accuracy_score(y_eval, pred))
    print(classification_report(y_eval, pred, digits=4, zero_division=0))
    print("Confusion:\n", confusion_matrix(y_eval, pred))

# ---- Save both ----
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
dump({"stageA": stageA, "stageB": stageB}, MODELS_DIR / "xgb_2stage_diabetes_gpu.joblib")
print("\nSaved:", MODELS_DIR / "xgb_2stage_diabetes_gpu.joblib")


===== VAL (2-stage) =====
Accuracy: 0.6357615894039735
              precision    recall  f1-score   support

           0     0.9571    0.6356    0.7639     32056
           1     0.0282    0.1960    0.0493       694
           2     0.3082    0.6941    0.4269      5302

    accuracy                         0.6358     38052
   macro avg     0.4312    0.5086    0.4134     38052
weighted avg     0.8497    0.6358    0.7039     38052

Confusion:
 [[20376  3802  7878]
 [  176   136   382]
 [  738   884  3680]]

===== TEST (2-stage) =====
Accuracy: 0.63786397561232
              precision    recall  f1-score   support

           0     0.9564    0.6386    0.7658     32055
           1     0.0300    0.1986    0.0521       695
           2     0.3041    0.6911    0.4224      5302

    accuracy                         0.6379     38052
   macro avg     0.4302    0.5094    0.4134     38052
weighted avg     0.8486    0.6379    0.7049     38052

Confusion:
 [[20470  3580  8005]
 [  179   138   37

In [5]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix

def predict_012_thresholds(X_in, threshold_risk=0.40, threshold_stageB=0.50):
    p_risk = stageA.predict_proba(X_in)[:, 1]
    is_risk = (p_risk >= threshold_risk)

    out = np.zeros(len(X_in), dtype=int)
    if np.any(is_risk):
        idx = np.where(is_risk)[0]
        X_r = X_in.iloc[idx]
        p_b = stageB.predict_proba(X_r)[:, 1]  # prob class 2
        out_r = np.where(p_b >= threshold_stageB, 2, 1)
        out[is_risk] = out_r
    return out

thresholds = np.round(np.arange(0.25, 0.71, 0.05), 2)

rows = []
for t in thresholds:
    pred = predict_012_thresholds(X_val, threshold_risk=t, threshold_stageB=0.50)
    f1_macro = f1_score(y_val, pred, average="macro", zero_division=0)
    rec1 = recall_score(y_val, pred, labels=[1], average=None, zero_division=0)[0]
    rec2 = recall_score(y_val, pred, labels=[2], average=None, zero_division=0)[0]
    prec1 = precision_score(y_val, pred, labels=[1], average=None, zero_division=0)[0]
    prec2 = precision_score(y_val, pred, labels=[2], average=None, zero_division=0)[0]
    acc = (pred == y_val.values).mean()
    rows.append((t, acc, f1_macro, rec1, prec1, rec2, prec2))

print("thr_risk | acc | f1_macro | rec1 | prec1 | rec2 | prec2")
for r in rows:
    print(f"{r[0]:>7} | {r[1]:.4f} | {r[2]:.4f}   | {r[3]:.4f} | {r[4]:.4f}  | {r[5]:.4f} | {r[6]:.4f}")

thr_risk | acc | f1_macro | rec1 | prec1 | rec2 | prec2
   0.25 | 0.5178 | 0.3639   | 0.2608 | 0.0229  | 0.7275 | 0.2749
    0.3 | 0.5595 | 0.3820   | 0.2406 | 0.0245  | 0.7188 | 0.2850
   0.35 | 0.5977 | 0.3982   | 0.2219 | 0.0266  | 0.7084 | 0.2961
    0.4 | 0.6358 | 0.4134   | 0.1960 | 0.0282  | 0.6941 | 0.3082
   0.45 | 0.6718 | 0.4271   | 0.1758 | 0.0311  | 0.6697 | 0.3210
    0.5 | 0.7047 | 0.4393   | 0.1542 | 0.0336  | 0.6437 | 0.3370
   0.55 | 0.7358 | 0.4493   | 0.1282 | 0.0368  | 0.6113 | 0.3524
    0.6 | 0.7641 | 0.4550   | 0.0908 | 0.0352  | 0.5719 | 0.3717
   0.65 | 0.7927 | 0.4610   | 0.0663 | 0.0383  | 0.5224 | 0.3977
    0.7 | 0.8143 | 0.4605   | 0.0461 | 0.0403  | 0.4579 | 0.4266


In [6]:
# =========================
# TABULAR AUGMENTATION (noise/jitter + feature dropout) + XGBoost GPU (2.x) - ONE CELL
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from joblib import dump

from xgboost import XGBClassifier

# -------------------------
# 1) LOAD
# -------------------------
ROOT = Path.cwd()
CSV_PATH = ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv"
TARGET = "Diabetes_012"

df = pd.read_csv(CSV_PATH)
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())
print("Data shape:", df.shape)
print("\nDỮ LIỆU LÚC ĐẦU (full dataset):")
print(y.value_counts().sort_index())

# -------------------------
# 2) SPLIT
# -------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\nTrain/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("\nTRAIN trước tăng cường:")
print(y_train.value_counts().sort_index())

# -------------------------
# 3) DEFINE AUGMENTATION (tabular)
# -------------------------
# Chọn các cột "continuous-ish" để cộng nhiễu (nếu không chắc, cứ để auto theo số unique lớn)
# Auto: cột có > 10 giá trị khác nhau coi là continuous-ish
cont_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=True) > 10]
bin_cols  = [c for c in X_train.columns if X_train[c].nunique(dropna=True) <= 10]

def augment_tabular(
    X_src: pd.DataFrame,
    n_new: int,
    noise_std_cont: float = 0.15,     # độ mạnh nhiễu cho continuous (tính theo std của cột)
    dropout_prob: float = 0.05,       # xác suất tắt feature (set 0)
    flip_prob_bin: float = 0.01,      # xác suất lật bit cho binary (rất nhỏ để không phá dữ liệu)
    random_state: int = 42
) -> pd.DataFrame:
    """
    Tạo n_new mẫu mới bằng cách lấy mẫu lại từ X_src rồi:
    - continuous: cộng Gaussian noise theo std từng cột
    - binary/ordinal: (tuỳ chọn) lật bit rất nhẹ
    - feature dropout: set 0 ngẫu nhiên một số feature
    """
    rng = np.random.default_rng(random_state)
    idx = rng.integers(0, len(X_src), size=n_new)
    X_new = X_src.iloc[idx].copy()

    # Noise cho continuous
    if len(cont_cols) > 0:
        for c in cont_cols:
            col = X_src[c].astype(float)
            s = float(col.std(ddof=0)) if np.isfinite(col.std(ddof=0)) else 0.0
            if s > 0:
                X_new[c] = X_new[c].astype(float) + rng.normal(0, noise_std_cont * s, size=n_new)
                # clip về min/max train để không tạo giá trị "ảo"
                X_new[c] = np.clip(X_new[c], col.min(), col.max())

    # Flip rất nhẹ cho binary/ordinal
    if len(bin_cols) > 0 and flip_prob_bin > 0:
        for c in bin_cols:
            # chỉ lật với xác suất nhỏ
            mask = rng.random(n_new) < flip_prob_bin
            # cố gắng chỉ lật 0/1; nếu cột không phải 0/1 thì bỏ qua
            vals = X_new[c].values
            uniq = np.unique(X_src[c].dropna().values)
            if set(uniq.tolist()).issubset({0, 1}):
                vals[mask] = 1 - vals[mask]
                X_new[c] = vals

    # Feature dropout (set 0)
    if dropout_prob > 0:
        drop_mask = rng.random((n_new, X_new.shape[1])) < dropout_prob
        X_new.values[drop_mask] = 0

    return X_new

# -------------------------
# 4) OVERSAMPLE + AUGMENT for minority classes (chỉ TRAIN)
# -------------------------
Xtr = X_train.copy()
ytr = y_train.copy()

counts = ytr.value_counts().to_dict()
target_n = max(counts.values())  # cân bằng về bằng class lớn nhất (class 0)
# Nếu bạn không muốn cân bằng hoàn toàn (đỡ tăng nhiễu), giảm xuống ví dụ: target_n = int(0.6 * max(counts.values()))

aug_parts = [Xtr]
aug_labels = [ytr]

for cls in sorted(counts.keys()):
    n_cls = counts[cls]
    if n_cls < target_n:
        need = target_n - n_cls
        X_src = Xtr.loc[ytr == cls]
        # tạo dữ liệu mới từ lớp này
        X_new = augment_tabular(
            X_src,
            n_new=need,
            noise_std_cont=0.12,     # bạn có thể thử 0.08 -> 0.20
            dropout_prob=0.03,       # thử 0.02 -> 0.08
            flip_prob_bin=0.005,     # rất nhỏ
            random_state=42 + cls
        )
        y_new = pd.Series([cls] * need, index=X_new.index)
        aug_parts.append(X_new)
        aug_labels.append(y_new)

X_train_aug = pd.concat(aug_parts, axis=0).reset_index(drop=True)
y_train_aug = pd.concat(aug_labels, axis=0).reset_index(drop=True).astype(int)

print("\nDỮ LIỆU SAU TĂNG CƯỜNG (TRAIN augmented):")
print(y_train_aug.value_counts().sort_index())
print("Train augmented shape:", X_train_aug.shape)

# -------------------------
# 5) PREPROCESS
# -------------------------
num_cols = X_train_aug.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols)],
    remainder="drop"
)

# -------------------------
# 6) TRAIN XGBOOST GPU (không cần sample_weight vì đã cân bằng bằng augment)
# -------------------------
num_classes = int(np.unique(y_train_aug).size)

xgb = XGBClassifier(
    n_estimators=2200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=2,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=42
)

model = Pipeline(steps=[("prep", preprocess), ("clf", xgb)])
model.fit(X_train_aug, y_train_aug)

# -------------------------
# 7) EVALUATE
# -------------------------
def eval_split(name, Xs, ys):
    pred = model.predict(Xs)
    proba = model.predict_proba(Xs)
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(ys, pred))
    print(classification_report(ys, pred, digits=4, zero_division=0))
    print("Confusion:\n", confusion_matrix(ys, pred))
    try:
        print("ROC-AUC (OVR):", roc_auc_score(ys, proba, multi_class="ovr"))
    except Exception as e:
        print("ROC-AUC không tính được:", e)

eval_split("VAL", X_val, y_val)
eval_split("TEST", X_test, y_test)

# -------------------------
# 8) SAVE
# -------------------------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_augmented_gpu.joblib"
dump(model, out_path)
print("\nSaved model:", out_path)

ROOT: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code
CSV_PATH: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv | exists: True
Data shape: (253680, 22)

DỮ LIỆU LÚC ĐẦU (full dataset):
Diabetes_012
0    213703
1      4631
2     35346
Name: count, dtype: int64

Train/Val/Test: (177576, 21) (38052, 21) (38052, 21)

TRAIN trước tăng cường:
Diabetes_012
0    149592
1      3242
2     24742
Name: count, dtype: int64

DỮ LIỆU SAU TĂNG CƯỜNG (TRAIN augmented):
0    149592
1    149592
2    149592
Name: count, dtype: int64
Train augmented shape: (448776, 21)

===== VAL =====
Accuracy: 0.8478923578261327
              precision    recall  f1-score   support

           0     0.8646    0.9747    0.9164     32056
           1     0.0000    0.0000    0.0000       694
           2     0.5338    0.1922    0.2826      5302

    accuracy                         0.8479     38052
   macro avg  

In [7]:
# =========================
# BINARY: 0 vs (1+2)  |  XGBoost GPU (XGBoost 2.x)  - ONE CELL
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, average_precision_score, precision_recall_curve
)

from xgboost import XGBClassifier
from joblib import dump

# ---------- 1) PATH + LOAD ----------
ROOT = Path.cwd()  # bạn đang ở ...\Code
CSV_PATH = ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv"
TARGET = "Diabetes_012"

df = pd.read_csv(CSV_PATH)
assert TARGET in df.columns, f"Không thấy cột target '{TARGET}'"

# ---------- 2) MAKE BINARY LABEL ----------
# risk = 1 nếu Diabetes_012 là 1 hoặc 2
y = (df[TARGET].astype(int) != 0).astype(int)
X = df.drop(columns=[TARGET])

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())
print("Data shape:", df.shape)
print("\nLABEL BINARY (0=No risk, 1=Risk):")
print(y.value_counts().rename({0:"NoRisk", 1:"Risk"}))

# ---------- 3) SPLIT ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\nTrain/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Train label:", y_train.value_counts().to_dict())

# ---------- 4) PREPROCESS ----------
num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols)],
    remainder="drop"
)

# ---------- 5) HANDLE IMBALANCE (scale_pos_weight) ----------
# scale_pos_weight = (#negative / #positive) trong TRAIN
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
print("\nscale_pos_weight =", scale_pos_weight)

# ---------- 6) XGBOOST GPU (2.x) ----------
xgb = XGBClassifier(
    n_estimators=2500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    device="cuda",              # ✅ GPU
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

model = Pipeline(steps=[("prep", preprocess), ("clf", xgb)])
model.fit(X_train, y_train)

# ---------- 7) EVAL FUNCTION + THRESHOLD TUNING ----------
def eval_binary(name, Xs, ys, threshold=0.50):
    proba = model.predict_proba(Xs)[:, 1]
    pred = (proba >= threshold).astype(int)

    print(f"\n===== {name} (threshold={threshold}) =====")
    print("Accuracy:", accuracy_score(ys, pred))
    print("ROC-AUC:", roc_auc_score(ys, proba))
    print("PR-AUC (Average Precision):", average_precision_score(ys, proba))
    print(classification_report(ys, pred, digits=4, zero_division=0))
    print("Confusion:\n", confusion_matrix(ys, pred))

# Baseline threshold 0.5
eval_binary("VAL", X_val, y_val, threshold=0.50)

# ---- chọn threshold tối ưu theo F1 trên VAL (quét nhanh) ----
proba_val = model.predict_proba(X_val)[:, 1]
ths = np.round(np.arange(0.10, 0.91, 0.05), 2)

best = None
for t in ths:
    pred = (proba_val >= t).astype(int)
    # F1 cho class Risk (1)
    tp = ((pred == 1) & (y_val.values == 1)).sum()
    fp = ((pred == 1) & (y_val.values == 0)).sum()
    fn = ((pred == 0) & (y_val.values == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
    cand = (f1, t, precision, recall)
    if (best is None) or (cand[0] > best[0]):
        best = cand

print("\nBEST threshold on VAL (by F1 for Risk):")
print("F1, thr, precision, recall =", best)

best_thr = best[1]

# Evaluate with best threshold
eval_binary("VAL", X_val, y_val, threshold=best_thr)
eval_binary("TEST", X_test, y_test, threshold=best_thr)

# ---------- 8) SAVE MODEL ----------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_binary_risk_cuda.joblib"
dump({"model": model, "best_threshold": float(best_thr)}, out_path)
print("\nSaved:", out_path)

ROOT: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code
CSV_PATH: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv | exists: True
Data shape: (253680, 22)

LABEL BINARY (0=No risk, 1=Risk):
Diabetes_012
NoRisk    213703
Risk       39977
Name: count, dtype: int64

Train/Val/Test: (177576, 21) (38052, 21) (38052, 21)
Train label: {0: 149592, 1: 27984}

scale_pos_weight = 5.345626072041166

===== VAL (threshold=0.5) =====
Accuracy: 0.7324187953327026
ROC-AUC: 0.812207155448279
PR-AUC (Average Precision): 0.4464808938737237
              precision    recall  f1-score   support

           0     0.9380    0.7307    0.8215     32056
           1     0.3400    0.7417    0.4662      5996

    accuracy                         0.7324     38052
   macro avg     0.6390    0.7362    0.6438     38052
weighted avg     0.8437    0.7324    0.7655     38052

Confusion:
 [[23423  8633]
 [ 1549  4

In [10]:
!pip install -U xgboost

     -------------------------------------- 101.7/101.7 MB 8.3 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.0.4
    Uninstalling xgboost-3.0.4:
      Successfully uninstalled xgboost-3.0.4


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\letra\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\~gboost\\lib\\xgboost.dll'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# =========================
# XGBoost (OLD VERSION COMPAT) - Random Search (NO early stopping) + Best Threshold
# Binary Risk (0 vs 1+2) - ONE CELL
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report, accuracy_score
)

from xgboost import XGBClassifier
from joblib import dump

# --------- 1) Load ----------
ROOT = Path.cwd()
CSV_PATH = ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv"
TARGET = "Diabetes_012"

df = pd.read_csv(CSV_PATH)
y = (df[TARGET].astype(int) != 0).astype(int)  # binary risk
X = df.drop(columns=[TARGET])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols)],
    remainder="drop"
)

# imbalance weight
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
print("scale_pos_weight =", scale_pos_weight)

# transform once
Xtr = preprocess.fit_transform(X_train)
Xva = preprocess.transform(X_val)
Xte = preprocess.transform(X_test)

# --------- 2) Search space ----------
param_dist = {
    "max_depth": [3,4,5,6,7,8],
    "min_child_weight": [1,2,3,4,5,6],
    "subsample": [0.7,0.8,0.85,0.9,1.0],
    "colsample_bytree": [0.7,0.8,0.85,0.9,1.0],
    "gamma": [0, 0.5, 1.0, 2.0, 5.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
}

n_iter = 30  # tăng lên 50 nếu muốn
sampler = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

# --------- 3) Try a few n_estimators values (no early stop) ----------
# Thường 1200~3000 là ổn với lr=0.03
n_estimators_list = [1200, 1800, 2500, 3200]

best = None
best_model = None
best_params = None

for n_estimators in n_estimators_list:
    print(f"\n=== Testing n_estimators={n_estimators} ===")
    for i, p in enumerate(sampler, 1):
        clf = XGBClassifier(
            n_estimators=n_estimators,
            learning_rate=0.03,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            device="cuda",                # nếu máy bạn có GPU + bản xgb hỗ trợ param này
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            **p
        )

        clf.fit(Xtr, y_train)

        proba = clf.predict_proba(Xva)[:, 1]
        auc = roc_auc_score(y_val, proba)
        pr = average_precision_score(y_val, proba)

        score = (auc, pr)
        if (best is None) or (score > best):
            best = score
            best_model = clf
            best_params = {"n_estimators": n_estimators, **p}

        if i % 10 == 0:
            print(f"[{i:02d}/{n_iter}] AUC={auc:.5f} | PR-AUC={pr:.5f}")

print("\n===== BEST on VAL =====")
print("Best AUC, PR-AUC:", best)
print("Best params:", best_params)

# --------- 4) Best threshold on VAL (max F1 Risk) ----------
proba_val = best_model.predict_proba(Xva)[:, 1]
ths = np.linspace(0.05, 0.95, 181)

best_thr = 0.5
best_f1 = -1.0
for t in ths:
    pred = (proba_val >= t).astype(int)
    f1 = f1_score(y_val, pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = float(t)

print("\nBest threshold (VAL, max F1 Risk):", best_thr, "F1:", best_f1)

def eval_split(name, Xmat, ytrue, thr):
    proba = best_model.predict_proba(Xmat)[:, 1]
    pred = (proba >= thr).astype(int)
    print(f"\n===== {name} (thr={thr}) =====")
    print("Accuracy:", accuracy_score(ytrue, pred))
    print("ROC-AUC:", roc_auc_score(ytrue, proba))
    print("PR-AUC:", average_precision_score(ytrue, proba))
    print(classification_report(ytrue, pred, digits=4, zero_division=0))
    print("Confusion:\n", confusion_matrix(ytrue, pred))

eval_split("VAL", Xva, y_val, best_thr)
eval_split("TEST", Xte, y_test, best_thr)

# --------- 5) Save ----------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_binary_tuned_no_es.joblib"
dump(
    {"preprocess": preprocess, "booster": best_model, "best_threshold": best_thr, "best_params": best_params},
    out_path
)
print("\nSaved:", out_path)

scale_pos_weight = 5.345626072041166

=== Testing n_estimators=1200 ===
[10/30] AUC=0.81496 | PR-AUC=0.45000
[20/30] AUC=0.82183 | PR-AUC=0.46291
[30/30] AUC=0.82144 | PR-AUC=0.46176

=== Testing n_estimators=1800 ===
[10/30] AUC=0.81018 | PR-AUC=0.44242
[20/30] AUC=0.82045 | PR-AUC=0.46014
[30/30] AUC=0.81961 | PR-AUC=0.45905

=== Testing n_estimators=2500 ===
[10/30] AUC=0.80500 | PR-AUC=0.43360
[20/30] AUC=0.81882 | PR-AUC=0.45686
[30/30] AUC=0.81769 | PR-AUC=0.45561

=== Testing n_estimators=3200 ===
[10/30] AUC=0.80037 | PR-AUC=0.42571
[20/30] AUC=0.81742 | PR-AUC=0.45404
[30/30] AUC=0.81617 | PR-AUC=0.45319

===== BEST on VAL =====
Best AUC, PR-AUC: (0.8238301763608149, 0.46649530416104235)
Best params: {'n_estimators': 1200, 'subsample': 0.9, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'max_depth': 5, 'gamma': 5.0, 'colsample_bytree': 0.7}

Best threshold (VAL, max F1 Risk): 0.6549999999999999 F1: 0.49194530932719643

===== VAL (thr=0.6549999999999999) =====
Acc

In [13]:
# =========================
# FULL ONE-CELL: Load CSV -> Binary label -> Split -> Preprocess -> Focused Tuning (maximize AUC) -> Test eval -> Save
# (Compatible with older xgboost: no early_stopping / no callbacks)
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, accuracy_score
)

from xgboost import XGBClassifier
from joblib import dump

# --------- 1) LOAD ----------
ROOT = Path.cwd()  # bạn đang ở ...\Code
CSV_PATH = ROOT.parent / "Data" / "diabetes_012_health_indicators_BRFSS2015.csv"
TARGET = "Diabetes_012"

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())

df = pd.read_csv(CSV_PATH)

# Binary: 0 = NoRisk, 1 = Risk (1 or 2)
y = (df[TARGET].astype(int) != 0).astype(int)
X = df.drop(columns=[TARGET])

print("Data shape:", df.shape)
print("Binary label counts:", y.value_counts().to_dict())

# --------- 2) SPLIT ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

# --------- 3) PREPROCESS ----------
num_cols = X_train.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols)],
    remainder="drop"
)

# transform once
Xtr = preprocess.fit_transform(X_train)
Xva = preprocess.transform(X_val)
Xte = preprocess.transform(X_test)

# imbalance weight
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
print("scale_pos_weight =", scale_pos_weight)

# --------- 4) FOCUSED SEARCH AROUND YOUR BEST REGION ----------
param_dist = {
    "max_depth": [4,5,6],
    "min_child_weight": [4,5,6,7],
    "subsample": [0.85,0.9,0.95],
    "colsample_bytree": [0.6,0.7,0.8],
    "gamma": [3.0,5.0,7.0],
    "reg_lambda": [5.0,10.0,15.0],
    "reg_alpha": [0.0, 0.1, 0.5],
}

n_iter = 50
sampler = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

# Bạn có thể thử 2 cấu hình lr/estimators (thường cải thiện AUC)
candidates = [
    {"n_estimators": 3000, "learning_rate": 0.02},
    {"n_estimators": 2200, "learning_rate": 0.025},
]

best = None
best_model = None
best_params = None

for cfg in candidates:
    print(f"\n=== Tuning with n_estimators={cfg['n_estimators']} | lr={cfg['learning_rate']} ===")
    for i, p in enumerate(sampler, 1):
        clf = XGBClassifier(
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            # Nếu bản xgboost của bạn không nhận device="cuda" thì comment dòng này
            device="cuda",
            # Nếu bản xgboost cũ hơn, dùng gpu_hist:
            # tree_method="gpu_hist",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            **p
        )

        clf.fit(Xtr, y_train)

        proba = clf.predict_proba(Xva)[:, 1]
        auc = roc_auc_score(y_val, proba)
        pr = average_precision_score(y_val, proba)

        score = (auc, pr)
        if (best is None) or (score > best):
            best = score
            best_model = clf
            best_params = {**cfg, **p}

        if i % 10 == 0:
            print(f"[{i:02d}/{n_iter}] AUC={auc:.5f} | PR-AUC={pr:.5f}")

print("\n===== BEST on VAL =====")
print("Best (AUC, PR-AUC):", best)
print("Best params:", best_params)

# --------- 5) BEST THRESHOLD BY F1 ON VAL ----------
proba_val = best_model.predict_proba(Xva)[:, 1]
ths = np.linspace(0.05, 0.95, 181)

best_thr = 0.5
best_f1 = -1.0
for t in ths:
    pred = (proba_val >= t).astype(int)
    f1 = f1_score(y_val, pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = float(t)

print("\nBest threshold (VAL max F1):", best_thr, "F1:", best_f1)

# --------- 6) EVAL TEST ----------
proba_test = best_model.predict_proba(Xte)[:, 1]
pred_test = (proba_test >= best_thr).astype(int)

print("\n===== TEST FINAL =====")
print("Accuracy:", accuracy_score(y_test, pred_test))
print("ROC-AUC:", roc_auc_score(y_test, proba_test))
print("PR-AUC:", average_precision_score(y_test, proba_test))
print(classification_report(y_test, pred_test, digits=4, zero_division=0))
print("Confusion:\n", confusion_matrix(y_test, pred_test))

# --------- 7) SAVE ----------
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
out_path = MODELS_DIR / "xgb_binary_FOCUSED_TUNED.joblib"
dump(
    {"preprocess": preprocess, "booster": best_model, "best_threshold": best_thr, "best_params": best_params},
    out_path
)
print("\nSaved:", out_path)

ROOT: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code
CSV_PATH: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Data\diabetes_012_health_indicators_BRFSS2015.csv | exists: True
Data shape: (253680, 22)
Binary label counts: {0: 213703, 1: 39977}
Train/Val/Test: (177576, 21) (38052, 21) (38052, 21)
scale_pos_weight = 5.345626072041166

=== Tuning with n_estimators=3000 | lr=0.02 ===
[10/50] AUC=0.82376 | PR-AUC=0.46579
[20/50] AUC=0.82359 | PR-AUC=0.46507
[30/50] AUC=0.82272 | PR-AUC=0.46460
[40/50] AUC=0.82376 | PR-AUC=0.46621
[50/50] AUC=0.82390 | PR-AUC=0.46645

=== Tuning with n_estimators=2200 | lr=0.025 ===
[10/50] AUC=0.82379 | PR-AUC=0.46605
[20/50] AUC=0.82356 | PR-AUC=0.46521
[30/50] AUC=0.82258 | PR-AUC=0.46467
[40/50] AUC=0.82366 | PR-AUC=0.46646
[50/50] AUC=0.82388 | PR-AUC=0.46624

===== BEST on VAL =====
Best (AUC, PR-AUC): (0.8239499425871304, 0.46614451865918277)
Best params: {'n_estimators': 2200, 'learning_

In [14]:
from joblib import load
import pandas as pd

# Đổi đúng tên file bạn đang dùng:
MODEL_PATH = r"e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code\models\xgb_binary_FOCUSED_TUNED.joblib"

bundle = load(MODEL_PATH)

preprocess = bundle["preprocess"]
booster = bundle["booster"]
thr = float(bundle.get("best_threshold", bundle.get("threshold", 0.5)))
params = bundle.get("best_params", None)

print("Loaded:", MODEL_PATH)
print("Threshold:", thr)
if params:
    print("Params:", params)

def predict_risk(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    df_input: DataFrame có đúng 21 cột feature giống lúc train (không có cột Diabetes_012)
    """
    Xp = preprocess.transform(df_input)
    proba = booster.predict_proba(Xp)[:, 1]  # P(Risk)
    pred = (proba >= thr).astype(int)

    out = df_input.copy()
    out["risk_proba"] = proba
    out["risk_pred"] = pred
    out["risk_label"] = out["risk_pred"].map({0: "NoRisk", 1: "Risk"})
    return out[["risk_proba", "risk_pred", "risk_label"]]

Loaded: e:\MachineLearning\Dự đoán nguy cơ mắc bệnh tiểu đường(đã có data 250k dòng)\Code\models\xgb_binary_FOCUSED_TUNED.joblib
Threshold: 0.6499999999999999
Params: {'n_estimators': 2200, 'learning_rate': 0.025, 'subsample': 0.95, 'reg_lambda': 15.0, 'reg_alpha': 0.0, 'min_child_weight': 7, 'max_depth': 5, 'gamma': 7.0, 'colsample_bytree': 0.6}
